# 🎬 影片使用情境分析器（Colab 批次版・訓練特徵用）

**功能：**
- 遞迴掃描 Google Drive 上的專案資料夾（含 `*_content.txt` 即視為專案）
- 從每個專案的 `*_resource_log.csv` 讀取影片 URL
- **兩階段判斷（CV 初篩 → Gemini 精確量化複判）：**
  1. **第一階段｜CV 通用代表性初篩**：運用 OpenCV 進行動態影格擷取與基礎影像前處理（色彩空間轉換、邊緣運算），依畫面資訊量（邊緣密度、亮度）與跟前一張候選幀的差異程度，篩掉明顯無意義或重複的畫面，挑出具代表性的候選幀。**不依賴膚色等與族群、光線相關的特徵**
  2. **第二階段｜Gemini 多模態量化複判**：對候選幀逐一（分批）評估 0–100 分的「使用情境呈現分數」。**為了讓比例可以直接拿去做訓練特徵，這裡不做抽樣後的比例回推估計，而是用實際評分結果精確計數**——候選幀數多時只會分批送出、成本增加，但不會犧牲準確度
- 輸出彙整 CSV
- **支援續傳**：中途額度用完、斷線或當機中斷，重新執行同一格即可自動跳過已完成的影片、接著跑

**輸出欄位（訓練特徵）：**
`project, category, video_filename, source, url, download_method, usage_scene_ratio, usage_scene_soft_ratio, gemini_avg_score, gemini_coverage, total_duration, sampled_frames, cv_candidate_frames, gemini_checked_frames, gemini_scored_frames, gemini_confirmed_frames, gemini_api_calls, usage_frames, classifier_mode, status`

- **`usage_scene_ratio`（硬門檻版，精確計數）**：實際被 Gemini 判定為使用情境（分數 ≥ `GEMINI_USAGE_SCORE_THRESHOLD`）的幀數 ÷ 全部取樣幀數。回答「使用情境佔整支影片多少比例」，適合當作訓練模型的主要特徵
- **`usage_scene_soft_ratio`（連續版）**：所有已評分候選幀的分數總和(/100) ÷ 全部取樣幀數。保留分數的強弱資訊而非只是二分類，適合給對連續特徵敏感的模型（例如線性模型）多一個角度
- **`gemini_avg_score`**：有實際評到分的候選幀之平均分數（0–100），代表「有出現使用情境的畫面，平均而言有多明顯」——這是強度指標，不是涵蓋範圍
- **`gemini_coverage`**：候選幀中，實際成功取得 Gemini 分數的比例。**訓練前務必檢查這一欄**：數值明顯 <1 代表這支影片的候選幀沒有全部評到分（額度用盡、API 失敗等），`usage_scene_ratio` 可能被低估，建議篩掉或另外處理覆蓋率過低的樣本
- `cv_candidate_frames` / `gemini_checked_frames` / `gemini_scored_frames` / `gemini_confirmed_frames` / `gemini_api_calls`：兩階段流程的中間過程數字，方便除錯與確認額度使用狀況

**資料夾結構：**
```
ZecZec_Dataset_E&G/
└── 預購式專案/教育/成功/
    └── ES32_hohoka-01/          ← 含 *_content.txt → 視為專案
        ├── ES32_resource_log.csv
        ├── ES32_content.txt
        |——ES32_plans.txt
        |——ES32_FAQ.txt
        ├── ES32_images/
        └── ES32_videos/
```

## ① 安裝依賴套件

In [ ]:
# 基本套件
!pip install -q opencv-python-headless yt-dlp gdown google-generativeai pillow

# ✏️ 若要使用 CLIP 模式（較精準，需較長安裝時間），取消下行註解
# !pip install -q torch torchvision clip-by-openai

print('✅ 套件安裝完成')

## ② 掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive 已掛載')

## ③ 設定路徑與模式（依實際情況修改）

In [ ]:
import os
import getpass

# ✏️ 根資料夾路徑
ROOT_DIR = '/content/drive/MyDrive/ZecZec_Group_Data/ZecZec_Dataset_E&G'

# ✏️ 輸出 CSV 路徑
OUTPUT_CSV = '/content/drive/MyDrive/ZecZec_Group_Data/video_features_result.csv'


# ✏️ 分類模式：'rule'（輕量，無需 GPU，通用代表性篩選）或 'clip'（語意式，需安裝 torch）
# 此模式僅作為「第一階段 CV 初篩」使用，篩出候選幀後交由 Gemini 量化複判
CLASSIFIER_MODE = 'rule'

# ✏️ 取幀間隔秒數（越小越精準但越慢，也會增加候選幀數 → 增加 Gemini 呼叫量）
SAMPLE_INTERVAL_SEC = 2.0

# ✏️ 暫存影片目錄（下載後自動清除）
TMP_DIR = '/content/tmp_videos'


# ══════════════════════════════════════════════════════════
# Gemini 複判設定（第二階段：對 CV 初篩後的候選幀做量化評分）
# ══════════════════════════════════════════════════════════
# ✏️ 是否啟用 Gemini 複判；False 則僅用 CV 初篩結果（等同原本行為）
USE_GEMINI_VERIFY = True

# ✏️ Gemini API 金鑰：執行到這裡時會跳出輸入框貼上，避免金鑰以明文寫進 notebook
GEMINI_API_KEY = getpass.getpass('請輸入 Gemini API Key（若不啟用複判可直接按 Enter 略過）：').strip() if USE_GEMINI_VERIFY else ''
if GEMINI_API_KEY:
    print(f'（已輸入 API Key，長度 {len(GEMINI_API_KEY)}，開頭 {GEMINI_API_KEY[:4]}***；'
          f'正常的 Gemini API Key 通常以 AIza 開頭、長度約 39 碼，若明顯不同請重新確認）')

# ✏️ Gemini 模型名稱（需支援圖片輸入）
GEMINI_MODEL = 'gemini-2.5-flash'

# ✏️ 每支影片最多送幾張候選幀給 Gemini
# 【訓練用資料重要】這裡不再用「抽樣＋比例回推」的方式估計，而是實際逐一評分、
# 用真實計數算 usage_scene_ratio，才不會把估計誤差帶進訓練特徵。
# 因此建議設高一點（例如 60），讓 CV 篩出的候選幀盡量都能被 Gemini 實際檢查到；
# 若擔心額度/時間，可搭配下面的 GEMINI_MAX_CALLS_PER_RUN 與續傳機制，分多次執行完成。
GEMINI_MAX_FRAMES_PER_VIDEO = 60

# ✏️ 【最小化呼叫次數關鍵】每次 Gemini 請求最多夾帶幾張候選幀（合併成一次請求評分）
# 預設等於 GEMINI_MAX_FRAMES_PER_VIDEO → 每支影片只送出「1 次」Gemini 呼叫
# （只要候選幀的 JPEG 總大小不超過 inline data 的 20MB 上限即可，一般影片截圖遠低於此）
# 想在單支影片內分批（例如避免單次請求失敗就整批重來），可調小此值，例如 5
GEMINI_BATCH_SIZE = 15

# ✏️ 每次呼叫 Gemini 後的間隔秒數（免費額度通常以「每分鐘請求數」限制，此為保底值；
# 若 Gemini 回傳的錯誤訊息中有建議等待秒數，程式會優先採用該建議值）
GEMINI_RATE_LIMIT_SEC = 5.0

# ✏️ 【最小化呼叫次數關鍵】整個執行過程中，Gemini API 最多呼叫幾次
# 超過後自動停止呼叫 Gemini、其餘候選幀直接 fallback 為只用 CV 初篩結果，
# 避免一次跑整批資料就把免費額度用光；請依你的方案額度上限調整
GEMINI_MAX_CALLS_PER_RUN = 40

# ✏️ Gemini 情境呈現分數（0–100）達此門檻，該幀視為「確認使用情境」，用來計算 usage_scene_ratio
GEMINI_USAGE_SCORE_THRESHOLD = 50


# ══════════════════════════════════════════════════════════
# 小規模測試模式（建議先開著跑一輪，確認額度與流程正常再關掉跑全部）
# ══════════════════════════════════════════════════════════
# ✏️ True：只處理前 TEST_MAX_PROJECTS 個專案、每個專案前 TEST_MAX_VIDEOS_PER_PROJECT 支影片
TEST_MODE = True
TEST_MAX_PROJECTS = 2
TEST_MAX_VIDEOS_PER_PROJECT = 1


# ══════════════════════════════════════════════════════════
# 續傳設定（額度用完、斷線、當機中斷後，重新執行同一格可以接著跑）
# ══════════════════════════════════════════════════════════
# ✏️ True：執行前讀取 OUTPUT_CSV 裡已經是 status=ok 的影片，自動跳過不重跑
RESUME_FROM_EXISTING = True

# ✏️ True：先前跑失敗（status 以 error 開頭）的影片，這次會重新嘗試；
#    False：連失敗過的也直接跳過（適合你已經確認過失敗原因、不想再重試的情況）
RETRY_FAILED_VIDEOS = True

print(f'根資料夾   : {ROOT_DIR}')
print(f'根資料夾存在: {os.path.isdir(ROOT_DIR)}')
print(f'輸出路徑   : {OUTPUT_CSV}')
print(f'分類模式   : {CLASSIFIER_MODE}（第一階段 CV 通用代表性初篩）')
print(f'Gemini 複判: {"啟用" if USE_GEMINI_VERIFY else "停用"}（batch={GEMINI_BATCH_SIZE}，'
      f'單支影片上限 {GEMINI_MAX_FRAMES_PER_VIDEO} 幀，全程呼叫上限 {GEMINI_MAX_CALLS_PER_RUN} 次）')
print(f'測試模式   : {"啟用（僅跑 " + str(TEST_MAX_PROJECTS) + " 個專案 x " + str(TEST_MAX_VIDEOS_PER_PROJECT) + " 支影片）" if TEST_MODE else "停用（跑全部資料）"}')
print(f'續傳設定   : RESUME_FROM_EXISTING={RESUME_FROM_EXISTING}, RETRY_FAILED_VIDEOS={RETRY_FAILED_VIDEOS}')

## ④ 載入核心函式

In [ ]:
import re
import csv
import math
import time
import json as _json
import tempfile
import logging
import subprocess
from enum import Enum
from dataclasses import dataclass
from typing import Optional

import cv2
import numpy as np

logging.basicConfig(level=logging.WARNING)  # Colab 內只顯示警告以上
logger = logging.getLogger(__name__)

# ── CLIP 可選依賴（第一階段 CV 初篩可選用）───────────────────
try:
    import torch
    import clip as openai_clip
    HAS_CLIP = True
except ImportError:
    HAS_CLIP = False

# ── Gemini 可選依賴（第二階段量化複判）───────────────────────
try:
    import google.generativeai as genai
    from PIL import Image
    HAS_GEMINI = True
except ImportError:
    HAS_GEMINI = False

_gemini_model = None
_gemini_calls_made = 0          # 全程已呼叫 Gemini API 的次數（含重試）
_gemini_budget_exhausted_logged = False


# ══════════════════════════════════════════════════════════
# 設定
# ══════════════════════════════════════════════════════════
class ClassifierMode(Enum):
    CLIP = 'clip'
    RULE = 'rule'

USAGE_SCENE_PROMPTS = [
    'a person using the product',
    'a person operating a device',
    'a product demonstration in real life',
    'someone holding and using the product',
    'the product being used in everyday life',
]
NON_USAGE_SCENE_PROMPTS = [
    'a product on a white background',
    'a product specification sheet',
    'text and graphics on screen',
    'an animated infographic',
    'a logo or brand identity',
    'talking head interview',
]

# ── 第一階段 CV 通用代表性初篩門檻（與使用情境語意無關，不含膚色特徵）──
CV_MIN_EDGE_DENSITY     = 0.03   # 邊緣密度過低 → 畫面資訊量太少（近乎純色卡/空景），排除
CV_MAX_BRIGHTNESS       = 245    # 平均亮度過高 → 過曝白畫面，排除
CV_MIN_BRIGHTNESS       = 10     # 平均亮度過低 → 近乎全黑畫面，排除
CV_DEDUP_DIFF_THRESHOLD = 8.0    # 與前一張候選幀的平均像素差異 → 過低視為重複靜態畫面，排除


# ══════════════════════════════════════════════════════════
# YouTube 下載
# ══════════════════════════════════════════════════════════
def extract_youtube_id(url: str) -> Optional[str]:
    patterns = [
        r'(?:v=|youtu\.be/|embed/)([A-Za-z0-9_-]{11})',
        r'shorts/([A-Za-z0-9_-]{11})',
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    return None

def download_youtube_video(url: str, output_dir: str, max_height: int = 360) -> Optional[str]:
    output_template = os.path.join(output_dir, '%(id)s.%(ext)s')
    cmd = [
        'yt-dlp',
        '--format', f'bestvideo[height<={max_height}][ext=mp4]+bestaudio[ext=m4a]/best[height<={max_height}][ext=mp4]/best',
        '--output', output_template,
        '--no-playlist', '--quiet', url,
    ]
    try:
        subprocess.run(cmd, check=True, timeout=180)
        for f in os.listdir(output_dir):
            if f.endswith(('.mp4', '.webm', '.mkv')):
                return os.path.join(output_dir, f)
    except Exception as e:
        logger.warning(f'yt-dlp 失敗: {e}')
    return None


# ══════════════════════════════════════════════════════════
# 幀取樣
# ══════════════════════════════════════════════════════════
def sample_frames(video_path: str, interval_sec: float) -> tuple[list[np.ndarray], float]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f'無法開啟影片: {video_path}')
    fps            = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    total_duration = total_frames / fps
    step           = max(1, int(fps * interval_sec))
    frames, idx    = [], 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            frames.append(frame)
        idx += 1
    cap.release()
    return frames, total_duration


# ══════════════════════════════════════════════════════════
# 第一階段：CV 通用代表性初篩（不含膚色特徵）
# ══════════════════════════════════════════════════════════
class RuleFrameSelector:
    """
    只依畫面本身的資訊量（邊緣密度、亮度）與跟前一張候選幀的差異程度，
    篩掉明顯無意義或重複的畫面，藉此避免用膚色比例等會受族群、光線影響
    的特徵去預判使用情境 —— 這一階段完全不判斷「是否為使用情境」，
    只挑出值得送去給 Gemini 判斷的代表性候選幀（同時也直接降低了要送給
    Gemini 的幀數，是控制 API 呼叫量的第一道關卡）。
    """
    def __init__(self):
        self._last_candidate_gray = None

    def reset(self):
        """開始分析新影片前呼叫，避免跨影片比對前一張候選幀。"""
        self._last_candidate_gray = None

    def is_representative_frame(self, frame_bgr: np.ndarray) -> bool:
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        mean_brightness = gray.mean()
        if mean_brightness > CV_MAX_BRIGHTNESS or mean_brightness < CV_MIN_BRIGHTNESS:
            return False

        edges = cv2.Canny(gray, 50, 150)
        edge_density = np.count_nonzero(edges) / edges.size
        if edge_density < CV_MIN_EDGE_DENSITY:
            return False

        if self._last_candidate_gray is not None:
            diff = cv2.absdiff(gray, self._last_candidate_gray).mean()
            if diff < CV_DEDUP_DIFF_THRESHOLD:
                return False

        self._last_candidate_gray = gray
        return True


class ClipSceneClassifier:
    """
    語意式初篩（選用，需 GPU/torch）：用 CLIP 對使用情境／非使用情境的
    文字提示做相似度比對，挑出語意上疑似使用情境的候選幀。
    """
    def __init__(self):
        if not HAS_CLIP:
            raise ImportError('請先安裝 CLIP')
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model, self.preprocess = openai_clip.load('ViT-B/32', device=self.device)
        self.model.eval()
        all_prompts = USAGE_SCENE_PROMPTS + NON_USAGE_SCENE_PROMPTS
        tokens = openai_clip.tokenize(all_prompts).to(self.device)
        with torch.no_grad():
            self.text_features = self.model.encode_text(tokens)
            self.text_features /= self.text_features.norm(dim=-1, keepdim=True)
        self.n_usage = len(USAGE_SCENE_PROMPTS)

    def reset(self):
        """無跨幀狀態，維持介面一致，不需做任何事。"""
        pass

    def is_representative_frame(self, frame_bgr: np.ndarray) -> bool:
        from PIL import Image
        rgb   = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        img_t = self.preprocess(Image.fromarray(rgb)).unsqueeze(0).to(self.device)
        with torch.no_grad():
            feat  = self.model.encode_image(img_t)
            feat /= feat.norm(dim=-1, keepdim=True)
            probs = (feat @ self.text_features.T).squeeze(0).softmax(dim=-1).cpu().numpy()
        return bool(probs[:self.n_usage].sum() > probs[self.n_usage:].sum())


# ══════════════════════════════════════════════════════════
# 第二階段：Gemini 多模態量化複判（批次請求 + 全域呼叫預算）
# ══════════════════════════════════════════════════════════
def init_gemini():
    """
    初始化 Gemini 模型（僅在 USE_GEMINI_VERIFY=True 時呼叫一次）。
    會立刻送一個極短的測試請求驗證金鑰是否有效，避免要跑到候選幀階段
    （每幀重試 2-3 次）才發現金鑰是錯的、浪費時間跟重試次數。
    """
    global _gemini_model
    if not HAS_GEMINI:
        raise ImportError('請先安裝 google-generativeai：!pip install -q google-generativeai')
    if not GEMINI_API_KEY:
        raise ValueError('請先設定 GEMINI_API_KEY')

    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel(GEMINI_MODEL)

    try:
        model.generate_content('ping')  # 最小測試請求，只為了驗證金鑰/模型是否可用
    except Exception as e:
        msg = (
            f'Gemini API 金鑰驗證失敗：{e}\n'
            f'請確認：\n'
            f'  1. 金鑰是從 https://aistudio.google.com/apikey 建立的「Gemini API」金鑰\n'
            f'     （不是 Google Cloud Console 一般用途的 API 金鑰，那種金鑰預設不會開通 Generative Language API）\n'
            f'  2. 貼上時沒有多貼到空白、換行或引號\n'
            f'  3. 若剛建立金鑰，稍等 1-2 分鐘讓它在後台生效後再試一次'
        )
        raise ValueError(msg) from e

    _gemini_model = model
    return _gemini_model


def gemini_budget_remaining() -> int:
    """回傳本次執行還剩多少次 Gemini 呼叫額度（依 GEMINI_MAX_CALLS_PER_RUN）。"""
    return max(0, GEMINI_MAX_CALLS_PER_RUN - _gemini_calls_made)


def _parse_retry_delay(err: Exception, default: float) -> float:
    """從 429 錯誤訊息中解析 Google 建議的等待秒數（例如 'retry in 39.9s'），沒有則用預設值。"""
    m = re.search(r'retry in\s+([\d.]+)\s*s', str(err), flags=re.IGNORECASE)
    if m:
        try:
            return float(m.group(1)) + 1.0  # 多留 1 秒緩衝
        except ValueError:
            pass
    return default


GEMINI_BATCH_SCORE_PROMPT_TEMPLATE = """你是產品短影音的畫面審核員。以下依序提供 {n} 張影片截圖（依序為第 1 張到第 {n} 張）。

請針對「每一張」畫面各自評估其呈現「產品實際使用情境」的程度，以 0 到 100 之間的整數分數表示。
「使用情境」請從寬認定，只要有人與產品直接互動、讓產品發揮功能，都算，包括但不限於：
- 手持、操作、配戴、按壓、組裝、示範功能等「操作型」使用
- 保養品／化妝品：塗抹、按摩、拍打在臉部或身體肌膚上
- 貼片、面膜、繃帶等：撕開包裝後貼在皮膚上、撕下使用前後的動作
- 服飾、飾品、鞋款等：穿戴在身上、實際走動展示上身效果
- 食品、飲品：實際咀嚼、飲用、烹調過程

請給分依據：
- 100 分：畫面清楚呈現上述任一種「人與產品直接互動使用」的動作
- 0 分：完全沒有使用情境，例如：產品純特寫、空景陳列、去背產品照（無人使用）、規格表、文字圖卡、動畫資訊圖、LOGO 或品牌識別畫面、純訪談口播（人臉講話但沒有碰觸或使用產品）
- 中間分數依畫面呈現使用情境的清楚程度給分（例如只有手部局部入鏡但看得出正在塗抹/貼上，可給中等分數）

請針對每一張畫面獨立判斷，不要跟其他張比較或排序，每張的分數只反映該張畫面本身的內容。

請只回答一個 JSON 陣列，長度必須剛好等於 {n}，依序對應每一張截圖的分數，例如：[80, 0, 45]
不要有任何其他文字、說明或標點。"""


def gemini_score_batch(frames_bgr_list: list, retries: int = 2) -> "Optional[list]":
    """
    【最小化 API 呼叫的核心】一次請求夾帶多張候選幀（最多 GEMINI_BATCH_SIZE 張），
    請 Gemini 一次回傳整批分數，取代逐幀呼叫。
    回傳與輸入等長的分數 list；若呼叫失敗、額度用盡或回應格式不對，回傳 None。
    """
    global _gemini_calls_made, _gemini_budget_exhausted_logged

    if _gemini_model is None or not frames_bgr_list:
        return None

    if gemini_budget_remaining() <= 0:
        if not _gemini_budget_exhausted_logged:
            logger.warning(f'已達本次執行 Gemini 呼叫上限（GEMINI_MAX_CALLS_PER_RUN={GEMINI_MAX_CALLS_PER_RUN}），'
                            f'其餘候選幀將直接 fallback 為只用 CV 初篩結果。')
            _gemini_budget_exhausted_logged = True
        return None

    n = len(frames_bgr_list)
    contents = [GEMINI_BATCH_SCORE_PROMPT_TEMPLATE.format(n=n)]
    for fb in frames_bgr_list:
        rgb = cv2.cvtColor(fb, cv2.COLOR_BGR2RGB)
        contents.append(Image.fromarray(rgb))

    for attempt in range(retries + 1):
        if gemini_budget_remaining() <= 0:
            return None
        _gemini_calls_made += 1
        try:
            resp = _gemini_model.generate_content(contents)
            text = (resp.text or '').strip()
            m = re.search(r'\[[^\[\]]*\]', text, flags=re.DOTALL)
            if not m:
                raise ValueError(f'無法從回應中解析 JSON 陣列：{text[:80]!r}')
            scores = _json.loads(m.group())
            if len(scores) != n:
                raise ValueError(f'分數數量（{len(scores)}）與圖片數量（{n}）不符')
            return [max(0.0, min(100.0, float(s))) for s in scores]
        except Exception as e:
            wait = _parse_retry_delay(e, default=GEMINI_RATE_LIMIT_SEC * (attempt + 1))
            logger.warning(f'Gemini 批次複判失敗（第 {attempt + 1} 次，{n} 張）：{e}；等待 {wait:.1f}s 後重試')
            time.sleep(wait)
    return None


# ══════════════════════════════════════════════════════════
# 結果資料類別
# ══════════════════════════════════════════════════════════
@dataclass
class VideoFeatureResult:
    usage_scene_ratio:       float             # 硬門檻版：confirmed / sampled_frames（精確計數，不做抽樣回推）
    usage_scene_soft_ratio:  float             # 連續版：sum(score/100) / sampled_frames，供訓練模型使用更平滑的特徵
    gemini_avg_score:        Optional[float]   # 有實際評分的候選幀之平均分數（0-100，強度指標）
    gemini_coverage:         Optional[float]   # 候選幀中，實際成功取得 Gemini 分數的比例（資料品質指標）
    total_duration:          float
    sampled_frames:          int
    cv_candidate_frames:     int
    gemini_checked_frames:   int               # 實際送 Gemini 嘗試評分的候選幀數
    gemini_scored_frames:    int               # 其中成功拿到分數的幀數（<= gemini_checked_frames）
    gemini_confirmed_frames: int               # 分數達門檻、視為使用情境的幀數
    gemini_api_calls:        int
    usage_frames:            int
    classifier_mode:         str


# ══════════════════════════════════════════════════════════
# 單支影片分析（CV 通用代表性初篩 → Gemini 批次量化複判 兩階段）
# ══════════════════════════════════════════════════════════
def analyze_video(video_path: str, classifier, interval_sec: float,
                   use_gemini_verify: bool = False,
                   gemini_max_frames: int = 60,
                   gemini_batch_size: int = 5,
                   gemini_score_threshold: float = 50.0,
                   debug_save_dir: "Optional[str]" = None) -> VideoFeatureResult:
    """
    第一階段（CV 初篩）：只依畫面資訊量與差異程度挑出代表性候選幀，不預判是否為使用情境。
    第二階段（Gemini 複判）：對候選幀逐一（分批）評分。

    【訓練用資料的關鍵設計】所有比例都用「實際評分結果」直接計數，不做抽樣後的比例回推：
      - usage_scene_ratio      = 實際被 Gemini 判定為使用情境的幀數 / 全部取樣幀數（精確值，非估計）
      - usage_scene_soft_ratio = 所有已評分候選幀的分數總和(/100) / 全部取樣幀數（連續值，保留分數的強弱資訊）
      - gemini_avg_score       = 已評分候選幀的平均分數（強度指標，只看有評到分的幀）
      - gemini_coverage        = 候選幀中實際成功評分的比例（資料品質指標；<1 代表有候選幀因額度或
                                  API 失敗而沒被評到，訓練時可用來篩掉覆蓋率過低、不可靠的樣本）
    若某候選幀最終沒能取得分數（額度用盡或多次重試失敗），一律視為「未評分」而不是「確認使用」或
    「不是使用」，避免用假設污染訓練特徵；這類情況只會反映在 gemini_coverage 偏低上。
    """
    calls_before = _gemini_calls_made
    frames, duration = sample_frames(video_path, interval_sec)
    total = len(frames)

    # ── 第一階段：CV 通用代表性初篩 ────────────────────────
    classifier.reset()
    cv_candidates = []
    for i, frame in enumerate(frames):
        try:
            if classifier.is_representative_frame(frame):
                cv_candidates.append(i)
        except Exception as e:
            logger.warning(f'第 {i} 幀 CV 初篩失敗: {e}')
    cv_candidate_count = len(cv_candidates)

    scores_by_index = {}

    # ── 第二階段：Gemini 批次量化複判（逐一評分，不做抽樣回推）──
    if not use_gemini_verify or cv_candidate_count == 0 or _gemini_model is None:
        gemini_checked_count   = 0
        gemini_scored_count    = 0
        gemini_confirmed_count = 0
        gemini_avg_score       = None
        gemini_coverage        = None
    else:
        to_check = cv_candidates
        if len(to_check) > gemini_max_frames:
            # 候選幀數超過單支影片上限，仍用等距抽樣縮減「要評分的數量」（成本控制），
            # 但這只影響 gemini_coverage（會反映覆蓋率下降），比例計算本身不再用這個子集反推全體
            step     = len(to_check) / gemini_max_frames
            to_check = [to_check[int(i * step)] for i in range(gemini_max_frames)]

        scores_by_index = {}
        for batch_start in range(0, len(to_check), gemini_batch_size):
            batch_idx = to_check[batch_start:batch_start + gemini_batch_size]
            if gemini_budget_remaining() <= 0:
                break  # 全域預算用盡，剩餘候選幀直接留白（未評分），不做任何假設
            batch_scores = gemini_score_batch([frames[i] for i in batch_idx])
            if batch_scores is None:
                continue  # 這批失敗 → 這些幀維持「未評分」
            for idx, sc in zip(batch_idx, batch_scores):
                scores_by_index[idx] = sc

        gemini_checked_count   = len(to_check)
        gemini_scored_count    = len(scores_by_index)
        gemini_confirmed_count = sum(1 for sc in scores_by_index.values() if sc >= gemini_score_threshold)
        gemini_avg_score       = round(sum(scores_by_index.values()) / gemini_scored_count, 2) if gemini_scored_count else None
        gemini_coverage        = round(gemini_scored_count / cv_candidate_count, 4) if cv_candidate_count else None

    # ── 最終特徵：精確計數，不做抽樣回推 ──────────────────────
    if total > 0:
        usage_scene_ratio      = round(gemini_confirmed_count / total, 4)
        soft_score_sum         = sum(scores_by_index.values())
        usage_scene_soft_ratio = round((soft_score_sum / 100.0) / total, 4)
    else:
        usage_scene_ratio      = 0.0
        usage_scene_soft_ratio = 0.0

    # ── 偵錯用：印出逐幀分數、並可選存成圖檔供人工核對 ──────────
    if debug_save_dir is not None:
        print(f'   🔍 逐幀分數明細（共 {len(cv_candidates)} 張候選幀）：')
        if cv_candidates:
            os.makedirs(debug_save_dir, exist_ok=True)
        for idx in cv_candidates:
            sc = scores_by_index.get(idx)
            label = f'{sc:.1f}' if sc is not None else 'N/A（未評分）'
            print(f'      幀 #{idx:>4}  分數={label}')
            if sc is not None:
                fname = os.path.join(debug_save_dir, f'frame{idx:04d}_score{sc:03.0f}.jpg')
            else:
                fname = os.path.join(debug_save_dir, f'frame{idx:04d}_scoreNA.jpg')
            try:
                cv2.imwrite(fname, frames[idx])
            except Exception as e:
                logger.warning(f'儲存偵錯圖檔失敗（幀 {idx}）：{e}')
        if cv_candidates:
            print(f'   📁 候選幀已存到 {debug_save_dir}（檔名含分數，可在 Colab 左側檔案總管開圖核對）')

    return VideoFeatureResult(
        usage_scene_ratio=usage_scene_ratio,
        usage_scene_soft_ratio=usage_scene_soft_ratio,
        gemini_avg_score=gemini_avg_score,
        gemini_coverage=gemini_coverage,
        total_duration=round(duration, 2),
        sampled_frames=total,
        cv_candidate_frames=cv_candidate_count,
        gemini_checked_frames=gemini_checked_count,
        gemini_scored_frames=gemini_scored_count,
        gemini_confirmed_frames=gemini_confirmed_count,
        gemini_api_calls=_gemini_calls_made - calls_before,
        usage_frames=gemini_confirmed_count,
        classifier_mode=CLASSIFIER_MODE,
    )


# ══════════════════════════════════════════════════════════
# 資料夾掃描：找出專案
# ══════════════════════════════════════════════════════════
def is_project_dir(path: str) -> bool:
    """含 *_content.txt 即視為專案資料夾。"""
    return any(f.endswith('_content.txt') for f in os.listdir(path))

def get_category(dirpath: str, root_dir: str) -> str:
    """擷取從根資料夾到專案之間的中間層路徑。"""
    rel   = os.path.relpath(dirpath, root_dir).replace('\\', '/')
    parts = rel.split('/')
    return '/'.join(parts[:-1]) if len(parts) > 1 else ''

# 支援的本地影片副檔名
VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.webm'}

def find_local_videos_folder(project_path: str) -> "Optional[str]":
    """尋找專案下的 *_videos/ 子資料夾，回傳路徑；找不到回傳 None。"""
    for item in os.listdir(project_path):
        if item.endswith('_videos') and os.path.isdir(os.path.join(project_path, item)):
            return os.path.join(project_path, item)
    return None

def read_log_video_entries(project_path: str) -> list:
    """讀取 *_resource_log.csv，回傳 type=Video 的所有列 {'filename', 'url'}。"""
    entries = []
    for fname in os.listdir(project_path):
        if not fname.endswith('_resource_log.csv'):
            continue
        csv_path = os.path.join(project_path, fname)
        with open(csv_path, 'r', encoding='utf-8-sig', errors='ignore') as f:
            reader = csv.DictReader(f)
            if not reader.fieldnames:
                continue
            fn_lower = {k.strip().lower(): k for k in reader.fieldnames}
            if 'type' not in fn_lower or 'url' not in fn_lower:
                continue
            type_col = fn_lower['type']
            url_col  = fn_lower['url']
            fn_col   = fn_lower.get('filename', None)
            for row in reader:
                if row[type_col].strip().lower() == 'video':
                    entries.append({
                        'filename': row[fn_col].strip() if fn_col else '',
                        'url':      row[url_col].strip(),
                    })
    return entries

def collect_video_entries(project_path: str) -> list:
    """
    影片來源優先順序：
      1. 本地 *_videos/ 資料夾（直接讀檔，每個影片檔為一筆）
      2. 若無本地資料夾或資料夾內無影片，改從 *_resource_log.csv 的 URL 下載

    回傳 [{'filename', 'local_path'(或None), 'url', 'source'}, ...]
    """
    videos_folder = find_local_videos_folder(project_path)

    # ── 優先：本地資料夾 ──────────────────────────────────
    if videos_folder:
        local_files = [
            f for f in os.listdir(videos_folder)
            if os.path.splitext(f)[1].lower() in VIDEO_EXTS
        ]
        if local_files:
            # 嘗試從 log 比對 URL（方便記錄，無則留空）
            log_entries = read_log_video_entries(project_path)
            log_url_map = {e['filename']: e['url'] for e in log_entries if e['filename']}
            return [
                {
                    'filename':   f,
                    'local_path': os.path.join(videos_folder, f),
                    'url':        log_url_map.get(f, ''),
                    'source':     'local',
                }
                for f in sorted(local_files)
            ]

    # ── 備援：從 log 取 URL ───────────────────────────────
    log_entries = read_log_video_entries(project_path)
    return [
        {
            'filename':   e['filename'],
            'local_path': None,
            'url':        e['url'],
            'source':     'url',
        }
        for e in log_entries if e['url']
    ]



# ══════════════════════════════════════════════════════════
# URL 統一下載入口  [Fix: resolve_url_to_video 補上定義]
# 支援：YouTube / Vimeo / 直連 mp4 / Google Drive / 其他平台
# ══════════════════════════════════════════════════════════
import urllib.request
import urllib.parse

def _is_direct_video_url(url: str) -> bool:
    """URL 結尾是影片副檔名 → 視為直連。"""
    path = urllib.parse.urlparse(url).path.lower()
    return any(path.endswith(ext) for ext in ('.mp4', '.mov', '.avi', '.webm', '.mkv'))


def _gdrive_direct_url(url: str) -> Optional[str]:
    """
    將 Google Drive 分享連結轉為直連下載 URL。
    支援格式：
      https://drive.google.com/file/d/{FILE_ID}/view
      https://drive.google.com/open?id={FILE_ID}
    """
    m = re.search(r'/file/d/([A-Za-z0-9_-]+)', url)
    if not m:
        m = re.search(r'[?&]id=([A-Za-z0-9_-]+)', url)
    if m:
        return f'https://drive.google.com/uc?export=download&id={m.group(1)}'
    return None


def _download_direct(url: str, output_dir: str) -> Optional[str]:
    """直連 URL（mp4 / Google Drive export）直接用 urllib 下載。"""
    try:
        parsed   = urllib.parse.urlparse(url)
        filename = os.path.basename(parsed.path) or 'video.mp4'
        if not os.path.splitext(filename)[1]:
            filename += '.mp4'
        dest = os.path.join(output_dir, filename)
        headers = {'User-Agent': 'Mozilla/5.0'}
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=120) as resp, \
             open(dest, 'wb') as out:
            out.write(resp.read())
        return dest if os.path.getsize(dest) > 1024 else None
    except Exception as e:
        logger.warning(f'直連下載失敗: {e}')
        return None


def resolve_url_to_video(url: str, output_dir: str) -> tuple[Optional[str], str]:
    """
    統一 URL 下載入口，回傳 (本地影片路徑, 下載方式說明)。
    失敗時回傳 (None, 失敗原因)。

    優先順序：
      1. YouTube / Shorts  → yt-dlp
      2. Vimeo / 其他平台  → yt-dlp（yt-dlp 支援 1000+ 平台）
      3. Google Drive      → urllib 直連 export URL
      4. 直連 mp4/mov/…   → urllib 直連
    """
    os.makedirs(output_dir, exist_ok=True)

    # ── 1. YouTube（含 Shorts）──────────────────────────────
    if extract_youtube_id(url):
        path = download_youtube_video(url, output_dir)
        if path:
            return path, 'yt-dlp/youtube'
        return None, 'yt-dlp/youtube 下載失敗（影片可能為私人或已下架）'

    # ── 2. Google Drive ─────────────────────────────────────
    if 'drive.google.com' in url:
        direct = _gdrive_direct_url(url)
        if direct:
            path = _download_direct(direct, output_dir)
            if path:
                return path, 'gdrive-direct'
        return None, 'Google Drive 下載失敗（請確認檔案為「任何人皆可檢視」）'

    # ── 3. 直連影片 URL ─────────────────────────────────────
    if _is_direct_video_url(url):
        path = _download_direct(url, output_dir)
        if path:
            return path, 'direct-url'
        return None, '直連 URL 下載失敗'

    # ── 4. 其他平台（Vimeo、Facebook 等）→ yt-dlp 嘗試 ─────
    path = download_youtube_video(url, output_dir)  # yt-dlp 通吃大多數平台
    if path:
        return path, 'yt-dlp/other'

    return None, f'所有下載方法均失敗（不支援的 URL 格式）: {url[:80]}'


print('✅ 核心函式載入完成（CV 通用代表性初篩 + Gemini 批次量化複判 兩階段流程）')

## ④-2　快速小規模測試（強烈建議：先跑這格，確認 Gemini 額度與流程正常再執行下方全量批次）

這格會**只挑一支影片**（優先用本地 `*_videos/` 裡的第一支，找不到才用 `resource_log.csv` 的第一個 URL）跑完整的兩階段流程，並印出：
- CV 初篩挑出幾張候選幀
- 這支影片實際打了幾次 Gemini API（受 `GEMINI_BATCH_SIZE` 影響，可用來估算全量跑下來大約會用掉多少額度）
- 每張候選幀的 Gemini 分數，方便判斷分數是否合理、門檻是否需要調整

確認沒問題、額度也還夠之後，再到下方設定 `TEST_MODE = False` 執行全量批次。

In [ ]:
# ── 快速小規模測試：只分析「一支」影片 ────────────────────
import itertools

test_video_path = None
test_video_label = None

for dirpath, dirnames, filenames in os.walk(ROOT_DIR):
    if dirpath == ROOT_DIR or not is_project_dir(dirpath):
        continue
    entries = collect_video_entries(dirpath)
    if not entries:
        continue
    entry = entries[0]
    if entry['source'] == 'local' and entry['local_path']:
        test_video_path  = entry['local_path']
        test_video_label = f"{os.path.basename(dirpath)} / {entry['filename']}（本地）"
    else:
        path, method = resolve_url_to_video(entry['url'], TMP_DIR)
        if path:
            test_video_path  = path
            test_video_label = f"{os.path.basename(dirpath)} / {entry['url'][:50]}（{method}）"
    if test_video_path:
        break

if not test_video_path:
    print('⚠️  掃描不到任何可用影片，請確認 ROOT_DIR 是否正確、資料夾內是否有影片。')
else:
    print(f'🎥 測試影片：{test_video_label}')

    # 初始化分類器（若正式跑批次前的 cell 已建立過，這裡重新建一份乾淨的即可）
    _test_classifier = ClipSceneClassifier() if (CLASSIFIER_MODE == 'clip' and HAS_CLIP) else RuleFrameSelector()

    if USE_GEMINI_VERIFY and _gemini_model is None:
        try:
            init_gemini()
        except Exception as e:
            print(f'⚠️  Gemini 初始化失敗：{e}（本次測試將只跑 CV 初篩）')

    calls_before = _gemini_calls_made
    result = analyze_video(
        test_video_path, _test_classifier, SAMPLE_INTERVAL_SEC,
        use_gemini_verify=USE_GEMINI_VERIFY,
        gemini_max_frames=GEMINI_MAX_FRAMES_PER_VIDEO,
        gemini_batch_size=GEMINI_BATCH_SIZE,
        gemini_score_threshold=GEMINI_USAGE_SCORE_THRESHOLD,
        debug_save_dir='/content/debug_frames',  # 逐幀分數 + 存圖，方便人工核對分數是否合理
    )

    print()
    print(f'   取樣幀數           : {result.sampled_frames}')
    print(f'   CV 候選幀數         : {result.cv_candidate_frames}')
    print(f'   Gemini 嘗試評分幀數   : {result.gemini_checked_frames}')
    print(f'   Gemini 成功評分幀數   : {result.gemini_scored_frames}（覆蓋率 gemini_coverage={result.gemini_coverage}）')
    print(f'   Gemini 確認使用情境幀數: {result.gemini_confirmed_frames}')
    print(f'   Gemini 平均分數      : {result.gemini_avg_score}')
    print(f'   本支影片實際呼叫次數   : {result.gemini_api_calls}  （batch size={GEMINI_BATCH_SIZE}）')
    print(f'   usage_scene_ratio     （硬門檻，精確計數）: {result.usage_scene_ratio:.2%}')
    print(f'   usage_scene_soft_ratio（連續分數版）      : {result.usage_scene_soft_ratio:.2%}')
    print()
    print(f'📊 本次測試總共消耗 Gemini 呼叫額度：{_gemini_calls_made - calls_before} 次'
          f'（剩餘預算 {gemini_budget_remaining()} / {GEMINI_MAX_CALLS_PER_RUN}）')
    print('💡 可依此推估：全量批次的預估呼叫次數 ≈ 專案數 × 每專案平均影片數 × 每支影片呼叫次數')

## ⑤ 執行批次分析並輸出 CSV

**續傳機制**：這格可以重複執行。若中途額度用完、斷線或當機中斷，直接重新執行本格即可——
程式會自動讀取 `OUTPUT_CSV` 裡已完成（`status=ok`）的影片並跳過，只繼續處理還沒跑過的部分。
每支影片分析完會立刻寫入並存檔，不會因為中斷而遺失已完成的結果。

In [ ]:
# ── 初始化分類器（第一階段：CV 通用代表性初篩）────────────
if CLASSIFIER_MODE == 'clip' and HAS_CLIP:
    classifier = ClipSceneClassifier()
    print('✅ 使用 CLIP 分類器（CV 初篩，語意式）')
else:
    if CLASSIFIER_MODE == 'clip':
        print('⚠️  CLIP 未安裝，自動降級為規則式代表性篩選器')
    classifier = RuleFrameSelector()
    print('✅ 使用規則式代表性篩選器（CV 初篩，不含膚色特徵）')

# ── 初始化 Gemini（第二階段：批次量化複判）────────────────
if USE_GEMINI_VERIFY:
    try:
        init_gemini()
        print(f'✅ Gemini 量化複判已啟用（模型：{GEMINI_MODEL}，batch={GEMINI_BATCH_SIZE}，'
              f'呼叫上限 {GEMINI_MAX_CALLS_PER_RUN} 次，門檻 {GEMINI_USAGE_SCORE_THRESHOLD} 分）')
    except Exception as e:
        print(f'⚠️  Gemini 初始化失敗，本次執行將僅使用 CV 初篩結果：{e}')
        USE_GEMINI_VERIFY = False
else:
    print('ℹ️  Gemini 複判未啟用，僅使用 CV 初篩結果')

if TEST_MODE:
    print(f'🧪 測試模式已啟用：僅處理前 {TEST_MAX_PROJECTS} 個專案、每個專案前 {TEST_MAX_VIDEOS_PER_PROJECT} 支影片')

os.makedirs(TMP_DIR, exist_ok=True)

# ── 輸出欄位 ──────────────────────────────────────────────
FIELDNAMES = [
    'project', 'category', 'video_filename', 'source',
    'url', 'download_method',
    'usage_scene_ratio', 'usage_scene_soft_ratio', 'gemini_avg_score', 'gemini_coverage',
    'total_duration', 'sampled_frames', 'cv_candidate_frames',
    'gemini_checked_frames', 'gemini_scored_frames', 'gemini_confirmed_frames', 'gemini_api_calls',
    'usage_frames', 'classifier_mode', 'status'
]


# ══════════════════════════════════════════════════════════
# 續傳機制
#   1. 執行前讀取 OUTPUT_CSV（若存在），把已經 status=ok（以及視設定決定的 error）
#      的影片記下來，等一下掃描到就直接跳過
#   2. 每分析完一支影片就立刻寫入 CSV 並 flush，而不是等全部跑完才一次寫入 ——
#      這樣即使額度用完、斷線、Colab 逾時中斷，已完成的部分也不會遺失，
#      下次重新執行這一格就能從中斷的地方接著跑
# ══════════════════════════════════════════════════════════
def _row_key(project, filename, url):
    return (project, (filename or url or '').strip())

output_exists  = os.path.isfile(OUTPUT_CSV)
completed_keys = set()

if RESUME_FROM_EXISTING and output_exists:
    try:
        with open(OUTPUT_CSV, 'r', newline='', encoding='utf-8-sig') as f:
            for r in csv.DictReader(f):
                status = (r.get('status') or '')
                if status == 'ok' or (status.startswith('error') and not RETRY_FAILED_VIDEOS):
                    completed_keys.add(_row_key(r.get('project', ''), r.get('video_filename', ''), r.get('url', '')))
        print(f'↩️  偵測到既有輸出檔（{OUTPUT_CSV}），其中 {len(completed_keys)} 筆已完成，執行時將自動跳過')
    except Exception as e:
        print(f'⚠️  讀取既有輸出檔失敗，將視為從頭開始：{e}')
        completed_keys = set()
        output_exists  = False
elif RESUME_FROM_EXISTING:
    print('ℹ️  尚無既有輸出檔，將從頭開始並建立新檔案')

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
csv_file   = open(OUTPUT_CSV, 'a', newline='', encoding='utf-8-sig')
csv_writer = csv.DictWriter(csv_file, fieldnames=FIELDNAMES)
if not output_exists:
    csv_writer.writeheader()
    csv_file.flush()

print('=' * 65)
print('     🎬 影片使用情境分析器（CV 通用初篩 + Gemini 批次量化複判版）')
print('=' * 65)
print(f'\n🔍 掃描根資料夾：{ROOT_DIR}\n')

all_rows      = []
project_count = 0
video_count   = 0
skipped_count = 0

try:
    for dirpath, dirnames, filenames in os.walk(ROOT_DIR):
        if dirpath == ROOT_DIR:
            continue
        if not is_project_dir(dirpath):
            continue
        if TEST_MODE and project_count >= TEST_MAX_PROJECTS:
            break

        project_name   = os.path.basename(dirpath)
        category       = get_category(dirpath, ROOT_DIR)
        project_count += 1

        print(f'\n📁 [{project_name}]  分類：{category}')

        video_entries = collect_video_entries(dirpath)

        if not video_entries:
            print(f'   ⚠️  找不到任何影片（本地及 URL 皆無），跳過')
            continue

        if TEST_MODE:
            video_entries = video_entries[:TEST_MAX_VIDEOS_PER_PROJECT]

        for entry in video_entries:
            filename   = entry['filename']
            local_path = entry['local_path']
            url        = entry['url']
            source     = entry['source']

            key = _row_key(project_name, filename, url)
            if RESUME_FROM_EXISTING and key in completed_keys:
                skipped_count += 1
                print(f'   ⏭️  已處理過（續傳跳過）：{filename or url[:55]}')
                continue

            video_count += 1
            src_label = '📂 本地' if source == 'local' else '🌐 URL'
            print(f'   🎥  {src_label}  {filename or url[:55]}')

            row = {
                'project':         project_name,
                'category':        category,
                'video_filename':  filename,
                'source':          source,
                'url':             url,
                'classifier_mode': CLASSIFIER_MODE,
                'download_method': '',
            }

            try:
                if source == 'local' and local_path:
                    # ── 本地影片：直接分析 ──────────────────────────────────
                    video_path      = local_path
                    download_method = 'local'
                else:
                    # ── URL：統一走 resolve_url_to_video ───────────────────
                    # 支援 YouTube、Vimeo、直連 mp4、Google Drive、其他平台
                    video_path, download_method = resolve_url_to_video(url, TMP_DIR)
                    if not video_path:
                        raise ValueError(
                            f'無法下載影片（方法：{download_method}）。'
                            f'請確認 URL 有效且為公開內容：{url[:80]}'
                        )
                    print(f'       ⬇️  下載完成（{download_method}）')

                row['download_method'] = download_method
                result = analyze_video(
                    video_path, classifier, SAMPLE_INTERVAL_SEC,
                    use_gemini_verify=USE_GEMINI_VERIFY,
                    gemini_max_frames=GEMINI_MAX_FRAMES_PER_VIDEO,
                    gemini_batch_size=GEMINI_BATCH_SIZE,
                    gemini_score_threshold=GEMINI_USAGE_SCORE_THRESHOLD,
                )

                row.update({
                    'usage_scene_ratio':       result.usage_scene_ratio,
                    'usage_scene_soft_ratio':  result.usage_scene_soft_ratio,
                    'gemini_avg_score':        result.gemini_avg_score if result.gemini_avg_score is not None else '',
                    'gemini_coverage':         result.gemini_coverage if result.gemini_coverage is not None else '',
                    'total_duration':          result.total_duration,
                    'sampled_frames':          result.sampled_frames,
                    'cv_candidate_frames':     result.cv_candidate_frames,
                    'gemini_checked_frames':   result.gemini_checked_frames,
                    'gemini_scored_frames':    result.gemini_scored_frames,
                    'gemini_confirmed_frames': result.gemini_confirmed_frames,
                    'gemini_api_calls':        result.gemini_api_calls,
                    'usage_frames':            result.usage_frames,
                    'status':                  'ok',
                })
                score_label = f'{result.gemini_avg_score:.1f}' if result.gemini_avg_score is not None else 'N/A'
                cov_label   = f'{result.gemini_coverage:.0%}' if result.gemini_coverage is not None else 'N/A'
                print(f'       CV 候選幀={result.cv_candidate_frames}  '
                      f'Gemini 評分={result.gemini_scored_frames}/{result.gemini_checked_frames}（覆蓋率 {cov_label}，確認 {result.gemini_confirmed_frames}，'
                      f'平均分數 {score_label}，實際呼叫 {result.gemini_api_calls} 次）  '
                      f'usage_scene_ratio={result.usage_scene_ratio:.2%}  soft_ratio={result.usage_scene_soft_ratio:.2%}  '
                      f'時長={result.total_duration:.1f}s')

                if USE_GEMINI_VERIFY and gemini_budget_remaining() <= 0:
                    print(f'       ⚠️  Gemini 呼叫額度已用盡（上限 {GEMINI_MAX_CALLS_PER_RUN} 次），'
                          f'之後影片將只使用 CV 初篩結果')

            except Exception as e:
                row.update({
                    'usage_scene_ratio':       '',
                    'usage_scene_soft_ratio':  '',
                    'gemini_avg_score':        '',
                    'gemini_coverage':         '',
                    'total_duration':          '',
                    'sampled_frames':          '',
                    'cv_candidate_frames':     '',
                    'gemini_checked_frames':   '',
                    'gemini_scored_frames':    '',
                    'gemini_confirmed_frames': '',
                    'gemini_api_calls':        '',
                    'usage_frames':            '',
                    'status':                  f'error: {e}',
                })
                print(f'       ❌ 失敗：{e}')

            # ── 續傳關鍵：每支影片一算完就立刻寫入並 flush ──────────────
            csv_writer.writerow(row)
            csv_file.flush()
            all_rows.append(row)

finally:
    csv_file.close()

if video_count > 0:
    print(f'\n✅ 完成！本次共分析 {video_count} 支影片（續傳跳過 {skipped_count} 支）')
    print(f'📁 結果已即時寫入：{OUTPUT_CSV}')
    print(f'📊 本次執行共呼叫 Gemini API {_gemini_calls_made} 次（上限 {GEMINI_MAX_CALLS_PER_RUN} 次）')
elif skipped_count > 0:
    print(f'\n✅ 所有符合條件的影片先前都已處理完成（跳過 {skipped_count} 支），沒有新的影片需要分析。')
else:
    print('\n⚠️  沒有找到任何影片資料。')

## ⑥ 預覽結果

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
print(f'共 {len(df)} 筆影片記錄')
df

## ⑦　彙總成專案層級候選特徵（先不合併，只產出多個候選欄位）

影片分析結果本身仍以「每支影片一列」為主（`OUTPUT_CSV` 不變）。這格另外**讀取同一份 CSV、依 `project` 分組**，
一次算出好幾種候選的專案層級彙總特徵，存成一份新的 CSV——先不跟你既有的 16 個特徵合併，
之後你決定好要用哪個（或哪幾個）候選特徵時，再自行用 `project` 當 key 去合併即可。

**彙總前的資料清理：**
- 只保留 `status == 'ok'`（下載/分析失敗的影片不納入）
- 篩掉 `gemini_coverage` 低於 `MIN_COVERAGE_FOR_AGG` 的影片（避免額度不足、評分不完整的影片拉低整體數字）

**產出的候選特徵：**
| 欄位 | 意義 |
|---|---|
| `primary_usage_scene_ratio` / `primary_usage_scene_soft_ratio` / `primary_gemini_avg_score` | 該專案「第一支影片」（依 CSV 蒐集順序，通常對應 `resource_log.csv`／頁面由上而下順序）的數值 |
| `mean_usage_scene_ratio` / `mean_usage_scene_soft_ratio` / `mean_gemini_avg_score` | 專案內所有影片的平均 |
| `max_usage_scene_ratio` / `max_usage_scene_soft_ratio` / `max_gemini_avg_score` | 專案內所有影片的最大值 |
| `duration_weighted_usage_scene_ratio` | 依各影片時長加權平均（影片越長，對整體印象影響越大） |
| `video_count` / `total_video_duration` | 專案影片支數／總時長，可當額外情境特徵 |
| `videos_included` / `videos_excluded_low_coverage` | 這次彙總實際用了幾支影片、又排除了幾支覆蓋率不足的影片，方便追蹤資料品質 |

哪個候選特徵最終有用，建議之後丟進 XGBoost 看 SHAP 重要性再決定，這格只負責產出，不做篩選。

In [ ]:
import pandas as pd

# ✏️ 彙總時，單支影片的 gemini_coverage 低於此門檻就不納入計算（避免評分不完整的影片拉低整體數字）
MIN_COVERAGE_FOR_AGG = 0.5

# ✏️ 彙總後輸出的專案層級候選特徵 CSV 路徑（獨立於 OUTPUT_CSV，先不與既有特徵合併）
PROJECT_LEVEL_CANDIDATES_CSV = os.path.join(
    os.path.dirname(OUTPUT_CSV), 'video_features_project_level_candidates.csv'
)

df = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')

# ── 只保留分析成功的列 ──────────────────────────────────
df_ok = df[df['status'] == 'ok'].copy()
for col in ['usage_scene_ratio', 'usage_scene_soft_ratio', 'gemini_avg_score',
            'gemini_coverage', 'total_duration']:
    df_ok[col] = pd.to_numeric(df_ok[col], errors='coerce')

agg_rows = []
for project, group in df_ok.groupby('project', sort=False):
    group_all = group.reset_index(drop=True)

    # 覆蓋率不足的影片先排除，但保留「排除前」的第一支影片當 primary 的候選來源
    primary_row = group_all.iloc[0]

    coverage_ok_mask = group_all['gemini_coverage'].isna() | (group_all['gemini_coverage'] >= MIN_COVERAGE_FOR_AGG)
    group = group_all[coverage_ok_mask]

    videos_included = len(group)
    videos_excluded = len(group_all) - videos_included

    if videos_included == 0:
        # 全部影片覆蓋率都不足，退而求其次至少保留 primary（供人工檢視，後續可自行決定是否採用）
        group = group_all.iloc[[0]]
        videos_included = 1
        videos_excluded = len(group_all) - 1

    total_dur = group['total_duration'].sum()
    if total_dur > 0:
        duration_weighted_ratio = (group['usage_scene_ratio'] * group['total_duration']).sum() / total_dur
    else:
        duration_weighted_ratio = group['usage_scene_ratio'].mean()

    agg_rows.append({
        'project':                         project,
        'category':                        group_all['category'].iloc[0],
        'video_count':                     len(group_all),
        'videos_included':                 videos_included,
        'videos_excluded_low_coverage':    videos_excluded,
        'total_video_duration':            round(total_dur, 2),

        'primary_usage_scene_ratio':       primary_row['usage_scene_ratio'],
        'primary_usage_scene_soft_ratio':  primary_row['usage_scene_soft_ratio'],
        'primary_gemini_avg_score':        primary_row['gemini_avg_score'],

        'mean_usage_scene_ratio':          round(group['usage_scene_ratio'].mean(), 4),
        'max_usage_scene_ratio':           round(group['usage_scene_ratio'].max(), 4),
        'mean_usage_scene_soft_ratio':     round(group['usage_scene_soft_ratio'].mean(), 4),
        'max_usage_scene_soft_ratio':      round(group['usage_scene_soft_ratio'].max(), 4),
        'mean_gemini_avg_score':           round(group['gemini_avg_score'].mean(), 2) if group['gemini_avg_score'].notna().any() else None,
        'max_gemini_avg_score':            round(group['gemini_avg_score'].max(), 2) if group['gemini_avg_score'].notna().any() else None,

        'duration_weighted_usage_scene_ratio': round(duration_weighted_ratio, 4),
    })

project_df = pd.DataFrame(agg_rows)
project_df.to_csv(PROJECT_LEVEL_CANDIDATES_CSV, index=False, encoding='utf-8-sig')

print(f'✅ 已彙總 {len(project_df)} 個專案的候選特徵')
print(f'📁 輸出至：{PROJECT_LEVEL_CANDIDATES_CSV}')
print(f'ℹ️  影片層級的原始資料（OUTPUT_CSV）維持不變，這份是額外的候選特徵表，尚未與既有特徵合併')

project_df